# 🎙️ TTS-Optimized Transcription Generator for Google Colab

**Generate Human-like Transcriptions with Prosodic Markers**

This notebook allows you to:
1. Upload your text file to transcribe
2. Select AI provider (Ollama or HuggingFace) and model
3. Choose language (auto-detect, Hindi, or English)
4. Generate TTS-optimized transcription with prosodic markers
5. Download the generated transcription

**What are Prosodic Markers?**
These special markers tell TTS models how to read the text naturally:
- `[PAUSE-SHORT]`, `[PAUSE-MEDIUM]`, `[PAUSE-LONG]` - Natural pauses
- `[TONE: thoughtful/curious/serious/calm/excited]` - Voice emotion
- `[EMPHASIS: word]` - Stress on specific words
- `[PACE: slow/normal/fast]` - Reading speed

**Recommended Models:**
- 🦙 Ollama: `gemma2:9b` (best for Hindi), `qwen2.5:7b`, `llama3.1:8b`
- 🤗 HuggingFace: `ai4bharat/Airavata` (Indian languages)

## 📦 Step 1: Install Dependencies
Run this cell to install all required packages.

In [11]:
# Install required packages
!pip install -q torch transformers accelerate sentencepiece
!pip install -q colorama huggingface-hub
!pip install -q ollama

print("\n🔐 HuggingFace Login (for gated models):")
print("   If using gated models, run the next cell to login.")
print("   Otherwise, skip to Step 2.\n")
print("✅ All dependencies installed!")


🔐 HuggingFace Login (for gated models):
   If using gated models, run the next cell to login.
   Otherwise, skip to Step 2.

✅ All dependencies installed!


### 🔐 (Optional) HuggingFace Login
Run this cell to login to HuggingFace for accessing gated models.

In [ ]:
# HuggingFace Login for Gated Models
from huggingface_hub import login, HfFolder
import os

print("🔐 HuggingFace Login Options:")
print("   1. Interactive Login - Opens a browser/token prompt")
print("   2. Token Login - Paste your HF token directly\n")

# Check if already logged in
existing_token = HfFolder.get_token()
if existing_token:
    print(f"✅ Already logged in to HuggingFace!")
    print(f"   Token: {existing_token[:10]}...{existing_token[-5:]}")
else:
    print("📝 Not logged in. Choose a login method:")
    print("\n🔹 Option 1: Interactive Login (recommended)")
    print("   Uncomment the line below and run this cell:")
    print("   # login()\n")
    print("🔹 Option 2: Token Login")
    print("   Get your token from: https://huggingface.co/settings/tokens")

# Uncomment ONE of the following lines to login:
# login()
# login(token="hf_YOUR_TOKEN_HERE")

### 🦙 (Optional) Ollama Setup
Run these cells if you want to use Ollama models. Skip if using HuggingFace only.

In [12]:
# Install and start Ollama server (required for Ollama models)
import subprocess
import time
import os

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

🦙 Installing Ollama...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

🚀 Starting Ollama server in background...
✅ Ollama server is running!


In [13]:
# Pull Ollama model (run this cell to select a model to download)
import ipywidgets as widgets
from IPython.display import display, HTML

print("🦙 Ollama Model Download")
print("=" * 50)

# Model selection for pulling
OLLAMA_MODELS_TO_PULL = {
    "gemma2:9b (Best for Hindi, ~5GB)": "gemma2:9b",
    "qwen2.5:7b (Balanced, ~4GB)": "qwen2.5:7b",
    "qwen2.5:3b (Fast, ~2GB)": "qwen2.5:3b",
    "aya:8b (Multilingual, ~5GB)": "aya:8b",
    "llama3.1:8b (English, ~5GB)": "llama3.1:8b",
    "mistral:7b (Quality, ~4GB)": "mistral:7b"
}

model_pull_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS_TO_PULL.keys()),
    value="gemma2:9b (Best for Hindi, ~5GB)",
    description='Model to Pull:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

custom_ollama_pull = widgets.Text(
    value='',
    placeholder='Or enter custom model name (e.g., aya-expanse:8b)',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_pull_dropdown)
display(custom_ollama_pull)
print("\n💡 Select a model and run the next cell to download it.")

🦙 Ollama Model Download


Dropdown(description='Model to Pull:', layout=Layout(width='400px'), options=('gemma2:9b (Best for Hindi, ~5GB…

Text(value='', description='Custom Model:', layout=Layout(width='400px'), placeholder='Or enter custom model n…


💡 Select a model and run the next cell to download it.


In [14]:
# Actually pull the selected model
import ollama

# Get model to pull
if custom_ollama_pull.value.strip():
    model_to_pull = custom_ollama_pull.value.strip()
else:
    model_to_pull = OLLAMA_MODELS_TO_PULL[model_pull_dropdown.value]

print(f"📥 Pulling model: {model_to_pull}")
print("   This may take several minutes depending on model size...\n")

try:
    # Pull with progress
    current_digest = ''
    for progress in ollama.pull(model_to_pull, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
            print()  # Newline between layers
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            completed = progress['completed']
            total = progress['total']
            pct = (completed / total * 100) if total > 0 else 0
            print(f"\r   {status}: {pct:.1f}% ({completed}/{total})", end='', flush=True)
        else:
            print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{model_to_pull}' pulled successfully!")

    # List available models
    print("\n📋 Available Ollama models:")
    models = ollama.list()
    for model in models.get('models', []):
        name = model.get('name', 'unknown')
        size = model.get('size', 0) / (1024**3)  # Convert to GB
        print(f"   • {name} ({size:.2f} GB)")

except Exception as e:
    print(f"\n❌ Error pulling model: {e}")
    print("   Make sure Ollama server is running (run the previous cell first).")

📥 Pulling model: llama3.1:8b
   This may take several minutes depending on model size...

   pulling 667b0c1932bc: 100.0% (4920738944/4920738944)
   pulling 948af2743fc7: 100.0% (1481/1481)
   pulling 0ba8f0e314b4: 100.0% (12320/12320)
   pulling 56bb8bd477a5: 100.0% (96/96)
   pulling 455f34728c9b: 100.0% (487/487)
   success

✅ Model 'llama3.1:8b' pulled successfully!

📋 Available Ollama models:
   • unknown (4.58 GB)


## 📤 Step 2: Upload Your Text File
Upload the text file you want to transcribe for TTS.

In [15]:
from google.colab import files
import os

print("📤 Please upload your text file to transcribe:")
uploaded = files.upload()

# Get the uploaded file name
UPLOADED_FILE = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {UPLOADED_FILE}")
print(f"📄 File size: {len(uploaded[UPLOADED_FILE])} bytes")

# Display preview
with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
    word_count = len(content.split())
    char_count = len(content)

print(f"\n📊 Content stats:")
print(f"   Words: {word_count:,}")
print(f"   Characters: {char_count:,}")
print(f"\n📝 Preview (first 500 chars):\n{content[:500]}...")

📤 Please upload your text file to transcribe:


Saving test_english.txt to test_english.txt

✅ Uploaded: test_english.txt
📄 File size: 170 bytes

📊 Content stats:
   Words: 26
   Characters: 170

📝 Preview (first 500 chars):
Hello world, this is a test of the speech synthesis system. We are checking if the text-to-speech functionality is working correctly with a simple English transcription.
...


## ⚙️ Step 3: Select AI Provider, Model & Language
Choose your preferred transcription model and settings.

In [16]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Provider options
PROVIDER_OPTIONS = {
    "Ollama (Recommended for TTS Transcription)": "ollama",
    "HuggingFace (GPU Accelerated)": "huggingface"
}

# Model options by provider
OLLAMA_MODEL_OPTIONS = {
    "gemma2:9b (Best for Hindi)": "gemma2:9b",
    "qwen2.5:7b (Balanced Quality)": "qwen2.5:7b",
    "qwen2.5:3b (Fast)": "qwen2.5:3b",
    "aya:8b (Multilingual Specialist)": "aya:8b",
    "llama3.1:8b (Best for English)": "llama3.1:8b",
    "Custom Model (enter below)": "custom"
}

HF_MODEL_OPTIONS = {
    "ai4bharat/Airavata (Best for Indian Languages)": "ai4bharat/Airavata",
    "sarvamai/sarvam-2b-v0.5 (Indian LLM)": "sarvamai/sarvam-2b-v0.5",
    "CohereForAI/aya-23-8B (Multilingual)": "CohereForAI/aya-23-8B",
    "Custom Model (enter below)": "custom"
}

# Language options
LANGUAGE_OPTIONS = {
    "Auto-detect": "auto",
    "Hindi (हिन्दी)": "hindi",
    "English": "english"
}

# Provider dropdown
provider_dropdown = widgets.Dropdown(
    options=list(PROVIDER_OPTIONS.keys()),
    value="Ollama (Recommended for TTS Transcription)",
    description='Provider:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

# Model dropdown (Ollama by default)
model_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODEL_OPTIONS.keys()),
    value="gemma2:9b (Best for Hindi)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

# Custom model input
custom_model_input = widgets.Text(
    value='',
    placeholder='Enter custom model name',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

# Language dropdown
language_dropdown = widgets.Dropdown(
    options=list(LANGUAGE_OPTIONS.keys()),
    value="Auto-detect",
    description='Language:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

# Chunk size slider
chunk_size_slider = widgets.IntSlider(
    value=6,
    min=3,
    max=12,
    step=1,
    description='Sentences/Chunk:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

display(HTML("<h3>🎛️ Configure Transcription Settings</h3>"))
display(provider_dropdown)
display(model_dropdown)
display(custom_model_input)
display(HTML("<br>"))
display(language_dropdown)
display(chunk_size_slider)

# Handle provider change to update model dropdown
def on_provider_change(change):
    if change['new'] == "HuggingFace (GPU Accelerated)":
        model_dropdown.options = list(HF_MODEL_OPTIONS.keys())
        model_dropdown.value = "ai4bharat/Airavata (Best for Indian Languages)"
        custom_model_input.placeholder = 'Enter HuggingFace model name'
    else:
        model_dropdown.options = list(OLLAMA_MODEL_OPTIONS.keys())
        model_dropdown.value = "gemma2:9b (Best for Hindi)"
        custom_model_input.placeholder = 'Enter Ollama model name'

provider_dropdown.observe(on_provider_change, names='value')
print("\n💡 Tip: gemma2:9b is recommended for Hindi TTS transcription!")
print("📝 Smaller chunk sizes = better quality but slower processing.")

Dropdown(description='Provider:', layout=Layout(width='450px'), options=('Ollama (Recommended for TTS Transcri…

Dropdown(description='Model:', layout=Layout(width='450px'), options=('gemma2:9b (Best for Hindi)', 'qwen2.5:7…

Text(value='', description='Custom Model:', layout=Layout(width='450px'), placeholder='Enter custom model name…

Dropdown(description='Language:', layout=Layout(width='350px'), options=('Auto-detect', 'Hindi (हिन्दी)', 'Eng…

IntSlider(value=6, description='Sentences/Chunk:', layout=Layout(width='350px'), max=12, min=3, style=SliderSt…


💡 Tip: gemma2:9b is recommended for Hindi TTS transcription!
📝 Smaller chunk sizes = better quality but slower processing.


In [17]:
# Store the selected configuration
SELECTED_PROVIDER = PROVIDER_OPTIONS[provider_dropdown.value]

# Get model based on provider
if SELECTED_PROVIDER == "huggingface":
    selected_model_key = model_dropdown.value
    SELECTED_MODEL = HF_MODEL_OPTIONS.get(selected_model_key, "custom")
else:
    SELECTED_MODEL = OLLAMA_MODEL_OPTIONS.get(model_dropdown.value, "custom")

if SELECTED_MODEL == "custom":
    SELECTED_MODEL = custom_model_input.value
    if not SELECTED_MODEL:
        raise ValueError("Please enter a custom model name!")

# Get language
SELECTED_LANGUAGE = LANGUAGE_OPTIONS[language_dropdown.value]
CHUNK_SIZE = chunk_size_slider.value

print(f"\n✅ Configuration saved:")
print(f"   🤖 Provider: {SELECTED_PROVIDER}")
print(f"   📦 Model: {SELECTED_MODEL}")
print(f"   🌐 Language: {SELECTED_LANGUAGE}")
print(f"   📦 Chunk Size: {CHUNK_SIZE} sentences/chunk")


✅ Configuration saved:
   🤖 Provider: ollama
   📦 Model: llama3.1:8b
   🌐 Language: english
   📦 Chunk Size: 6 sentences/chunk


## 🚀 Step 4: TTS Transcription Engine Setup
This cell contains the complete TTS-optimized transcription engine.

In [21]:
#!/usr/bin/env python3
"""
TTS-Optimized Transcription Engine for Google Colab
Generates transcriptions with prosodic markers for natural TTS audio
"""

import os
import sys
import json
import time
import warnings
import re
from pathlib import Path
from datetime import datetime
from collections import OrderedDict

warnings.filterwarnings("ignore")

# Check for available backends
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    HF_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    HF_AVAILABLE = False
    DEVICE = "cpu"

print(f"🖥️ Device: {DEVICE}")
if DEVICE == "cuda":
    import torch
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


# ============= TTS OPTIMIZED PROMPTS =============

SYSTEM_PROMPT_HINDI = """आप एक विशेषज्ञ TTS स्क्रिप्ट लेखक हैं। आपका काम टेक्स्ट को TTS-अनुकूल ट्रांसक्रिप्शन में बदलना है जो मानव-जैसी आवाज़ बनाएगा। आपके आउटपुट में केवल ट्रांसक्रिप्शन होना चाहिए, और कुछ नहीं।

**PROSODIC MARKERS जोड़ें**:
1. **PAUSES**: [PAUSE-SHORT]=0.3s, [PAUSE-MEDIUM]=0.6s, [PAUSE-LONG]=1.0s, [BREATH]
2. **TONE**: [TONE: thoughtful/curious/serious/calm/excited/mysterious/warm/dramatic]
3. **EMPHASIS**: [EMPHASIS: शब्द], [STRESS: शब्द]
4. **PACING**: [PACE: slow/normal/fast]

**GOLDEN RULES**:
1. ✅ मूल शब्दों को रखें - कुछ भी न बदलें
2. ✅ प्रासंगिक prosodic markers जोड़ें (3-5 प्रति वाक्य)
3. ✅ आउटपुट में केवल ट्रांसक्रिप्शन होना चाहिए (और कुछ नहीं)
4. ❌ कोई व्याख्या, सारांश या अतिरिक्त विवरण नहीं
5. ❌ कोई हेडर न लिखें - सीधे ट्रांसक्रिप्शन से शुरू करें"""

SYSTEM_PROMPT_ENGLISH = """You are an expert TTS script writer. Your job is to transform text into TTS-optimized transcription that will produce human-like voice. Your output should only contain transcription text, nothing else.

**ADD PROSODIC MARKERS**:
1. **PAUSES**: [PAUSE-SHORT]=0.3s, [PAUSE-MEDIUM]=0.6s, [PAUSE-LONG]=1.0s, [BREATH]
2. **TONE**: [TONE: thoughtful/curious/serious/calm/excited/mysterious/warm/dramatic]
3. **EMPHASIS**: [EMPHASIS: word], [STRESS: word]
4. **PACING**: [PACE: slow/normal/fast]

**GOLDEN RULES**:
1. ✅ Keep original words - change NOTHING
2. ✅ Add appropriate prosodic markers (3-5 per sentence)
3. ✅ Transcribe the COMPLETE input - do not skip any parts
4. ✅ Output should only contain transcribed text (nothing else)
5. ❌ NO interpretation, summary, or extra details
6. ❌ NO headers - start directly with transcription """

NARRATION_TEMPLATE_HINDI = """नीचे दिया गया टेक्स्ट को TTS-अनुकूल ट्रांसक्रिप्शन में बदलें।

**INPUT TEXT**:
\"\"\"
{text}
\"\"\"

अपना रिस्पांस सीधे ट्रांसक्रिप्शन से शुरू करें:"""

NARRATION_TEMPLATE_ENGLISH = """Transform the text below into TTS-optimized transcription.

**INPUT TEXT**:
\"\"\"
{text}
\"\"\"

Start your response directly with the transcription:"""


def detect_language(text):
    """Detect if text is primarily Hindi or English."""
    hindi_chars = len(re.findall(r'[\u0900-\u097F]', text))
    english_chars = len(re.findall(r'[a-zA-Z]', text))
    total_chars = hindi_chars + english_chars
    if total_chars == 0:
        return "english"
    hindi_ratio = hindi_chars / total_chars
    return "hindi" if hindi_ratio > 0.3 else "english"


def clean_llm_output(text):
    """Clean LLM output by removing headers and unwanted prefixes."""
    patterns_to_remove = [
        r'^\*\*TTS-OPTIMIZED TRANSCRIPTION\*\*:?\s*',
        r'^TTS-OPTIMIZED TRANSCRIPTION:?\s*',
        r'^\*\*TRANSCRIPTION\*\*:?\s*',
        r'^TRANSCRIPTION:?\s*',
        r'^\"\"\"?\s*',
        r'\"\"\"?\s*$',
    ]

    cleaned = text.strip()
    for pattern in patterns_to_remove:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE | re.MULTILINE)

    cleaned = cleaned.strip('"\'')
    return cleaned.strip()


def remove_repetitions(text):
    """Remove repeated sentences and phrases."""
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    seen = OrderedDict()

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        key = ' '.join(sent.split()[:10]).lower()
        if key not in seen:
            seen[key] = sent

    return ' '.join(seen.values())


def count_markers(transcription):
    """Count prosodic markers in transcription."""
    return {
        'pause': len(re.findall(r'\[PAUSE-', transcription)),
        'tone': len(re.findall(r'\[TONE:', transcription)),
        'emphasis': len(re.findall(r'\[EMPHASIS:', transcription)),
        'pace': len(re.findall(r'\[PACE:', transcription)),
        'breath': len(re.findall(r'\[BREATH\]', transcription))
    }


def split_into_sentences(text):
    """Split into sentences (Hindi + English)."""
    sentences = re.split(r'(?<=[.!?।])\s+(?=[A-ZА-Я"\u0900-\u097F])', text)
    return [s.strip() for s in sentences if s.strip()]


def create_chunks(sentences, chunk_size=6, overlap=1):
    """Create smaller overlapping chunks for better TTS quality."""
    chunks = []
    i = 0

    while i < len(sentences):
        chunk_sentences = sentences[i:i + chunk_size]
        chunk_text = ' '.join(chunk_sentences)

        chunks.append({
            'text': chunk_text,
            'start_idx': i,
            'end_idx': i + len(chunk_sentences)
        })

        i += max(1, chunk_size - overlap)

    return chunks


# ============= OLLAMA TRANSCRIPTION ENGINE =============

class OllamaTranscriptionEngine:
    """Transcription engine using Ollama local models."""

    def __init__(self, model_name, language="auto"):
        self.model_name = model_name
        self.language = language

        print(f"📥 Initializing Ollama engine with model: {model_name}")

        # Verify model is available
        try:
            import ollama
            self.client = ollama
            models = ollama.list()
            available = [m.get('name', '').split(':')[0] for m in models.get('models', [])]
            model_base = model_name.split(':')[0]

            if not any(model_base in m for m in available):
                print(f"⚠️ Model '{model_name}' not found. Attempting to pull...")
                ollama.pull(model_name)
                print(f"✅ Model '{model_name}' pulled successfully!")
            else:
                print(f"✅ Model '{model_name}' is available!")
        except Exception as e:
            print(f"❌ Error initializing Ollama: {e}")
            raise

    def transcribe(self, text):
        """Generate TTS-optimized transcription."""
        detected_lang = detect_language(text)
        lang = self.language if self.language != "auto" else detected_lang

        if lang == "hindi":
            system_prompt = SYSTEM_PROMPT_HINDI
            user_prompt = NARRATION_TEMPLATE_HINDI.format(text=text)
        else:
            system_prompt = SYSTEM_PROMPT_ENGLISH
            user_prompt = NARRATION_TEMPLATE_ENGLISH.format(text=text)

        try:
            response = self.client.generate(
                model=self.model_name,
                prompt=f"{system_prompt}\n\n{user_prompt}",
                options={
                    "temperature": 0.3,
                    "top_p": 0.9,
                    "num_predict": 2048,
                }
            )

            narration = response['response'].strip()
            narration = clean_llm_output(narration)
            narration = remove_repetitions(narration)
            markers = count_markers(narration)

            return narration, lang, markers

        except Exception as e:
            print(f"❌ Transcription error: {e}")
            return text, lang, {}


# ============= HUGGINGFACE TRANSCRIPTION ENGINE =============

class HuggingFaceTranscriptionEngine:
    """Transcription engine using HuggingFace models."""

    def __init__(self, model_name, language="auto", device="cuda"):
        self.model_name = model_name
        self.language = language
        self.device = device

        print(f"📥 Loading HuggingFace model: {model_name}")

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

            device_map = "auto" if device == "cuda" else None
            torch_dtype = torch.float16 if device == "cuda" else torch.float32

            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map=device_map,
                torch_dtype=torch_dtype,
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )

            if device == "cpu" and device_map is None:
                self.model = self.model.to(device)

            print("✅ HuggingFace model loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise

    def transcribe(self, text):
        """Generate TTS-optimized transcription."""
        detected_lang = detect_language(text)
        lang = self.language if self.language != "auto" else detected_lang

        if lang == "hindi":
            system_prompt = SYSTEM_PROMPT_HINDI
            user_prompt = NARRATION_TEMPLATE_HINDI.format(text=text)
        else:
            system_prompt = SYSTEM_PROMPT_ENGLISH
            user_prompt = NARRATION_TEMPLATE_ENGLISH.format(text=text)

        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]

            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.tokenizer(input_text, return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    temperature=0.3,
                    top_p=0.9,
                    do_sample=True
                )

            narration = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            narration = narration.split("assistant")[-1].strip()
            narration = clean_llm_output(narration)
            narration = remove_repetitions(narration)
            markers = count_markers(narration)

            return narration, lang, markers

        except Exception as e:
            print(f"❌ Transcription error: {e}")
            return text, lang, {}


# ============= MAIN TRANSCRIPTION GENERATOR =============

class TTSTranscriptionGenerator:
    """Main class for generating TTS-optimized transcriptions."""

    def __init__(self, provider, model_name, language="auto", output_dir=".", chunk_size=6):
        self.provider = provider
        self.model_name = model_name
        self.language = language
        self.output_dir = Path(output_dir)
        self.chunk_size = chunk_size

        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Initialize engine based on provider
        if provider == "ollama":
            self.engine = OllamaTranscriptionEngine(model_name, language)
        else:
            self.engine = HuggingFaceTranscriptionEngine(model_name, language, DEVICE)

    def generate_from_file(self, input_file):
        """Generate TTS-optimized transcription from file."""
        print("=" * 70)
        print("🎙️ TTS-OPTIMIZED TRANSCRIPTION GENERATOR")
        print("=" * 70)

        print(f"\n📖 Reading: {input_file}")
        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read().strip()

        primary_lang = detect_language(text)
        print(f"🌍 Detected language: {primary_lang.upper()}")
        print(f"🤖 Model: {self.model_name}")
        print(f"📦 Chunk size: {self.chunk_size} sentences")
        print("=" * 70)

        # Split into sentences and create chunks
        sentences = split_into_sentences(text)
        chunks = create_chunks(sentences, chunk_size=self.chunk_size, overlap=1)

        print(f"\n📦 Created {len(chunks)} chunks from {len(sentences)} sentences")
        print(f"\n🎯 STARTING TRANSCRIPTION\n")

        narrated_chunks = []
        total_markers = {'pause': 0, 'tone': 0, 'emphasis': 0, 'pace': 0, 'breath': 0}
        start_time = time.time()

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()

            print(f"\n{'=' * 50}")
            print(f"📄 Chunk {i}/{len(chunks)}")
            print(f"   Input: {len(chunk['text'].split())} words")

            narration, lang, markers = self.engine.transcribe(chunk['text'])

            # Update marker counts
            for key in total_markers:
                total_markers[key] += markers.get(key, 0)

            chunk_time = time.time() - chunk_start
            marker_str = f"P:{markers.get('pause',0)} T:{markers.get('tone',0)} E:{markers.get('emphasis',0)}"
            print(f"   ✅ [{lang}] {marker_str} ({chunk_time:.1f}s)")

            # Progress
            elapsed = time.time() - start_time
            avg = elapsed / i
            remaining = len(chunks) - i
            eta = remaining * avg
            print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

            narrated_chunks.append(narration)

        # Combine and save
        final_transcription = "\n\n".join(narrated_chunks)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        txt_file = self.output_dir / f"tts_transcription_{timestamp}.txt"

        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write(final_transcription)

        # Summary
        total_time = time.time() - start_time

        print(f"\n{'=' * 70}")
        print(f"🎉 TTS TRANSCRIPTION COMPLETE!")
        print(f"{'=' * 70}")
        print(f"⏱️ Time: {total_time/60:.2f} minutes")
        print(f"📦 Chunks: {len(chunks)}")
        print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
        print(f"\n🎭 Prosodic Markers Added:")
        print(f"   Pauses: {total_markers['pause']}")
        print(f"   Tones: {total_markers['tone']}")
        print(f"   Emphasis: {total_markers['emphasis']}")
        print(f"   Pace: {total_markers['pace']}")
        print(f"   Total: {sum(total_markers.values())}")
        print(f"\n💾 Output: {txt_file}")
        print(f"{'=' * 70}")

        return str(txt_file)


print("✅ TTS Transcription Engine loaded and ready!")

🖥️ Device: cpu
✅ TTS Transcription Engine loaded and ready!


## 🎙️ Step 5: Generate TTS Transcription
Run this cell to generate the TTS-optimized transcription.

In [22]:
# Create output directory
OUTPUT_DIR = "./transcription_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the generator
print("🚀 Initializing TTS Transcription Generator...")
print(f"   Provider: {SELECTED_PROVIDER}")
print(f"   Model: {SELECTED_MODEL}")

generator = TTSTranscriptionGenerator(
    provider=SELECTED_PROVIDER,
    model_name=SELECTED_MODEL,
    language=SELECTED_LANGUAGE,
    output_dir=OUTPUT_DIR,
    chunk_size=CHUNK_SIZE
)

# Generate transcription
print(f"\n🎙️ Starting transcription...")
OUTPUT_FILE = generator.generate_from_file(UPLOADED_FILE)

print(f"\n✅ Transcription file generated: {OUTPUT_FILE}")

🚀 Initializing TTS Transcription Generator...
   Provider: ollama
   Model: llama3.1:8b
📥 Initializing Ollama engine with model: llama3.1:8b
⚠️ Model 'llama3.1:8b' not found. Attempting to pull...
✅ Model 'llama3.1:8b' pulled successfully!

🎙️ Starting transcription...
🎙️ TTS-OPTIMIZED TRANSCRIPTION GENERATOR

📖 Reading: test_english.txt
🌍 Detected language: ENGLISH
🤖 Model: llama3.1:8b
📦 Chunk size: 6 sentences

📦 Created 1 chunks from 2 sentences

🎯 STARTING TRANSCRIPTION


📄 Chunk 1/1
   Input: 26 words
   ✅ [english] P:1 T:3 E:1 (190.7s)
   📈 Progress: 100.0% | ETA: 0.0m

🎉 TTS TRANSCRIPTION COMPLETE!
⏱️ Time: 3.18 minutes
📦 Chunks: 1
⚡ Avg/chunk: 190.7s

🎭 Prosodic Markers Added:
   Pauses: 1
   Tones: 3
   Emphasis: 1
   Pace: 1
   Total: 7

💾 Output: transcription_output/tts_transcription_20260129_101601.txt

✅ Transcription file generated: transcription_output/tts_transcription_20260129_101601.txt


## 📖 Step 6: Preview & Download Transcription
View your transcription and download it.

In [ ]:
from IPython.display import display, HTML
import os

if os.path.exists(OUTPUT_FILE):
    # Read and display transcription
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        transcription = f.read()

    file_size = os.path.getsize(OUTPUT_FILE) / 1024  # KB
    word_count = len(transcription.split())
    marker_count = sum(count_markers(transcription).values())

    print(f"📊 Transcription stats:")
    print(f"   Words: {word_count:,}")
    print(f"   Characters: {len(transcription):,}")
    print(f"   Prosodic markers: {marker_count}")
    print(f"   File size: {file_size:.2f} KB")

    print(f"\n📖 Preview (first 1500 chars):")
    print(f"{'=' * 60}")
    print(transcription[:1500])
    print(f"{'=' * 60}")
    if len(transcription) > 1500:
        print(f"... [truncated, {len(transcription) - 1500:,} more chars]")
else:
    print("❌ Output file not found. Please run the transcription step again.")

In [23]:
# Download the transcription file
from google.colab import files
import os

if os.path.exists(OUTPUT_FILE):
    print("📥 Downloading your TTS transcription file...")
    files.download(OUTPUT_FILE)
    print("\n✅ Download started! Check your browser downloads.")
    print("\n💡 This transcription is optimized for TTS models!")
    print("   Feed it to your TTS model for natural, human-like audio.")
else:
    print("❌ Output file not found. Please run the transcription step first.")

📥 Downloading your TTS transcription file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started! Check your browser downloads.

💡 This transcription is optimized for TTS models!
   Feed it to your TTS model for natural, human-like audio.


---

## 💡 Tips for Best Results

### Model Recommendations:
- **Hindi text**: Use `gemma2:9b` or `ai4bharat/Airavata`
- **English text**: Use `llama3.1:8b` or `qwen2.5:7b`
- **Multilingual**: Use `aya:8b`

### Chunk Size Guide:
- **3-4 sentences**: Highest quality, slowest
- **5-6 sentences**: Balanced (recommended)
- **8-10 sentences**: Faster, may miss some nuances

### Using the Output:
The generated transcription includes prosodic markers like:
- `[PAUSE-SHORT]` - 0.3 second pause
- `[PAUSE-MEDIUM]` - 0.6 second pause
- `[TONE: excited]` - Convey excitement
- `[EMPHASIS: word]` - Stress on a word

Feed this file to your TTS model (like Chatterbox, ParlerTTS, or others) for natural-sounding audio!